In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install torch_geometric "scanpy==1.11.4"

## import

In [ ]:
import torch

if torch.cuda.is_available():
    device = "cuda"
    print("GPU: ",torch.cuda.get_device_name(0))
else:
    device = "cpu"
    print("Using CPU")

Using CPU


In [ ]:
##Working with google colab
import os
import sys

os.chdir("/content/drive/MyDrive/Thesis/Projects/Master_Thesis/Notebooks/SelfAttention")
cwd = os.getcwd()
print(cwd)

sys.path.append("../../")

/content/drive/MyDrive/Thesis/Projects/Master_Thesis/Notebooks/SelfAttention


In [ ]:
import scanpy as sc
import numpy as np
import torch
from sklearn.preprocessing import Normalizer

import matplotlib.pyplot as plt

plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['mathtext.fontset'] = 'dejavuserif'
plt.rcParams['font.family'] = 'arial'

pltkw = dict(bbox_inches='tight', transparent=True)

/usr/local/lib/python3.12/dist-packages/scanpy/_utils/__init__.py:33: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  from anndata import __version__ as anndata_version
/usr/local/lib/python3.12/dist-packages/scanpy/__init__.py:24: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  if Version(anndata.__version__) >= Version("0.11.0rc2"):
/usr/local/lib/python3.12/dist-packages/scanpy/readwrite.py:16: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  if Version(anndata.__version__) >= Version("0.11.0rc2"):


In [ ]:
import SelfAttention as SA
import importlib

# Dataset

Founsation Models

In [ ]:
adata = sc.read_h5ad("/content/drive/MyDrive/Thesis/Projects/Data/Mouse_Brain/FMs_3/UNI_adata.h5ad")

In [ ]:
adata = sc.read_h5ad("/content/drive/MyDrive/Thesis/Projects/Data/Mouse_Brain/FMs_3/hoptimus_adata.h5ad")

In [ ]:
adata = sc.read_h5ad("/content/drive/MyDrive/Thesis/Projects/Data/Mouse_Brain/FMs_3/virchow_adata.h5ad")

Noise

In [ ]:
adata = sc.read_h5ad("../../../Data/Mouse_Coronal_Embeddings/Xenium_Morpho_integrated.h5ad")

In [ ]:
rng = np.random.default_rng(42)
ad = adata.copy()
ad.obsm["morpho"] = rng.standard_normal((63173, 500))

In [ ]:
adata.obsm['p_Morpho_Embedding'] = ad.obsm['morpho'] - np.min(ad.obsm['morpho'])

Preparing Dataset

In [ ]:
adata = SA.prep_adata(adata, norm=True, log1p=True)
data = SA.build_graph(adata, k=8, morph_key='p_Morpho_Embedding')

/usr/local/lib/python3.12/dist-packages/legacy_api_wrap/__init__.py:88: UserWarning: Some cells have zero counts
  return fn(*args_all, **kw)


In [ ]:
print(data.x.shape)              # shape of each node
print(data.edge_index[:, :10])   # first 10 edges
print(data.edge_attr[:10])       # first 10 edge features

torch.Size([63173, 5506])
tensor([[    0, 46082,     0, 46073,     0, 46072,     0, 46081,     0, 46079],
        [46082,     0, 46073,     0, 46072,     0, 46081,     0, 46079,     0]])
tensor([7.0088, 7.0088, 8.3267, 8.3267, 8.4724, 8.4724, 8.7292, 8.7292, 9.1721,
        9.1721])


In [ ]:
data.edge_index.shape

torch.Size([2, 1010768])

In [ ]:
505384.0/63173

8.0

In [ ]:
data.edge_attr.shape

torch.Size([1010768])

In [ ]:
importlib.reload(SA)
importlib.reload(SA.model)

<module 'SelfAttention.model' from '/content/drive/MyDrive/Thesis/Projects/Master_Thesis/Notebooks/SelfAttention/../../SelfAttention/model.py'>

In [ ]:
import gc

gc.collect()
torch.cuda.empty_cache()

# Train

In [ ]:
SA.set_random_seed(42)
model = SA.SpatialTransformerAE(
    in_dim=data.x.shape[1],
    gene_dim=data.gene_dim,
    hidden_dim=96,
    latent_dim=32,
    heads=4
)

### NOISE

In [ ]:
# NOISE
SA.model.fit(model, data, max_epochs=10000, mask_ratio=0.8, stop_eps=1e-7, stop_tol=200, device=device)

Epoch 0 | Loss: 0.2889
Best : inf
Epoch 10 | Loss: 0.1230
Best : 0.1232
Epoch 20 | Loss: 0.1209
Best : 0.1211
Epoch 30 | Loss: 0.1188
Best : 0.1190
Epoch 40 | Loss: 0.1167
Best : 0.1169
Epoch 50 | Loss: 0.1148
Best : 0.1150
Epoch 60 | Loss: 0.1130
Best : 0.1132
Epoch 70 | Loss: 0.1113
Best : 0.1115
Epoch 80 | Loss: 0.1098
Best : 0.1099
Epoch 90 | Loss: 0.1083
Best : 0.1084
Epoch 100 | Loss: 0.1070
Best : 0.1071
Epoch 110 | Loss: 0.1057
Best : 0.1058
Epoch 120 | Loss: 0.1046
Best : 0.1047
Epoch 130 | Loss: 0.1035
Best : 0.1036
Epoch 140 | Loss: 0.1025
Best : 0.1026
Epoch 150 | Loss: 0.1016
Best : 0.1016
Epoch 160 | Loss: 0.1007
Best : 0.1008
Epoch 170 | Loss: 0.0999
Best : 0.1000
Epoch 180 | Loss: 0.0991
Best : 0.0992
Epoch 190 | Loss: 0.0984
Best : 0.0985
Epoch 200 | Loss: 0.0978
Best : 0.0978
Epoch 210 | Loss: 0.0971
Best : 0.0972
Epoch 220 | Loss: 0.0965
Best : 0.0966
Epoch 230 | Loss: 0.0960
Best : 0.0960
Epoch 240 | Loss: 0.0955
Best : 0.0955
Epoch 250 | Loss: 0.0950
Best : 0.0950


In [ ]:
os.makedirs("saved_models", exist_ok=True)
model_dir = "saved_models"

torch.save(model.state_dict(), 'saved_models/mouse_brain_32_NOISE_0.8.pth')

### h_Optimus

In [ ]:
# h_optimus
SA.model.fit(model, data, max_epochs=10000, mask_ratio=0.8, stop_eps=1e-7, stop_tol=200, device=device)

Epoch 0 | Loss: 0.1284
Best : inf
Epoch 10 | Loss: 0.1207
Best : 0.1212
Epoch 20 | Loss: 0.1140
Best : 0.1147
Epoch 30 | Loss: 0.1059
Best : 0.1067
Epoch 40 | Loss: 0.0980
Best : 0.0987
Epoch 50 | Loss: 0.0916
Best : 0.0921
Epoch 60 | Loss: 0.0871
Best : 0.0875
Epoch 70 | Loss: 0.0841
Best : 0.0844
Epoch 80 | Loss: 0.0820
Best : 0.0822
Epoch 90 | Loss: 0.0804
Best : 0.0806
Epoch 100 | Loss: 0.0791
Best : 0.0792
Epoch 110 | Loss: 0.0778
Best : 0.0780
Epoch 120 | Loss: 0.0765
Best : 0.0766
Epoch 130 | Loss: 0.0751
Best : 0.0752
Epoch 140 | Loss: 0.0736
Best : 0.0738
Epoch 150 | Loss: 0.0722
Best : 0.0723
Epoch 160 | Loss: 0.0708
Best : 0.0709
Epoch 170 | Loss: 0.0695
Best : 0.0696
Epoch 180 | Loss: 0.0684
Best : 0.0685
Epoch 190 | Loss: 0.0673
Best : 0.0674
Epoch 200 | Loss: 0.0665
Best : 0.0665
Epoch 210 | Loss: 0.0657
Best : 0.0657
Epoch 220 | Loss: 0.0650
Best : 0.0651
Epoch 230 | Loss: 0.0645
Best : 0.0645
Epoch 240 | Loss: 0.0640
Best : 0.0640
Epoch 250 | Loss: 0.0636
Best : 0.0636


In [ ]:
os.makedirs("saved_models", exist_ok=True)
model_dir = "saved_models"

torch.save(model.state_dict(), 'saved_models/mouse_brain_32_hoptimus_0.8.pth')

### UNI

In [ ]:
# UNI
SA.model.fit(model, data, max_epochs=10000, mask_ratio=0.8, stop_eps=1e-7, stop_tol=200, device=device)

Epoch 0 | Loss: 0.1271
Best : inf
Epoch 10 | Loss: 0.1205
Best : 0.1211
Epoch 20 | Loss: 0.1133
Best : 0.1141
Epoch 30 | Loss: 0.1049
Best : 0.1058
Epoch 40 | Loss: 0.0967
Best : 0.0974
Epoch 50 | Loss: 0.0903
Best : 0.0908
Epoch 60 | Loss: 0.0862
Best : 0.0865
Epoch 70 | Loss: 0.0835
Best : 0.0837
Epoch 80 | Loss: 0.0816
Best : 0.0818
Epoch 90 | Loss: 0.0803
Best : 0.0804
Epoch 100 | Loss: 0.0791
Best : 0.0792
Epoch 110 | Loss: 0.0778
Best : 0.0780
Epoch 120 | Loss: 0.0764
Best : 0.0766
Epoch 130 | Loss: 0.0749
Best : 0.0750
Epoch 140 | Loss: 0.0733
Best : 0.0734
Epoch 150 | Loss: 0.0717
Best : 0.0719
Epoch 160 | Loss: 0.0704
Best : 0.0706
Epoch 170 | Loss: 0.0690
Best : 0.0691
Epoch 180 | Loss: 0.0678
Best : 0.0679
Epoch 190 | Loss: 0.0667
Best : 0.0668
Epoch 200 | Loss: 0.0658
Best : 0.0659
Epoch 210 | Loss: 0.0651
Best : 0.0651
Epoch 220 | Loss: 0.0644
Best : 0.0645
Epoch 230 | Loss: 0.0639
Best : 0.0640
Epoch 240 | Loss: 0.0634
Best : 0.0635
Epoch 250 | Loss: 0.0630
Best : 0.0631


In [ ]:
os.makedirs("saved_models", exist_ok=True)
model_dir = "saved_models"

torch.save(model.state_dict(), 'saved_models/mouse_brain_32_UNI_0.8.pth')

### virchow

In [ ]:
# virchow
SA.model.fit(model, data, max_epochs=10000, mask_ratio=0.8, stop_eps=1e-7, stop_tol=200, device=device)

Epoch 0 | Loss: 0.1288
Best : inf
Epoch 10 | Loss: 0.1208
Best : 0.1213
Epoch 20 | Loss: 0.1138
Best : 0.1146
Epoch 30 | Loss: 0.1053
Best : 0.1062
Epoch 40 | Loss: 0.0973
Best : 0.0981
Epoch 50 | Loss: 0.0910
Best : 0.0915
Epoch 60 | Loss: 0.0867
Best : 0.0871
Epoch 70 | Loss: 0.0839
Best : 0.0842
Epoch 80 | Loss: 0.0820
Best : 0.0822
Epoch 90 | Loss: 0.0807
Best : 0.0808
Epoch 100 | Loss: 0.0795
Best : 0.0796
Epoch 110 | Loss: 0.0784
Best : 0.0785
Epoch 120 | Loss: 0.0772
Best : 0.0773
Epoch 130 | Loss: 0.0758
Best : 0.0759
Epoch 140 | Loss: 0.0742
Best : 0.0743
Epoch 150 | Loss: 0.0725
Best : 0.0726
Epoch 160 | Loss: 0.0708
Best : 0.0710
Epoch 170 | Loss: 0.0693
Best : 0.0695
Epoch 180 | Loss: 0.0681
Best : 0.0682
Epoch 190 | Loss: 0.0669
Best : 0.0670
Epoch 200 | Loss: 0.0660
Best : 0.0661
Epoch 210 | Loss: 0.0653
Best : 0.0654
Epoch 220 | Loss: 0.0647
Best : 0.0647
Epoch 230 | Loss: 0.0642
Best : 0.0642
Epoch 240 | Loss: 0.0637
Best : 0.0638
Epoch 250 | Loss: 0.0633
Best : 0.0633


In [ ]:
os.makedirs("saved_models", exist_ok=True)
model_dir = "saved_models"

torch.save(model.state_dict(), 'saved_models/mouse_brain_32_virchow_0.8.pth')